# FinOps Starter — Validation & Idempotency Checks

Acceptance checks:
1. Two consecutive pipeline runs; the second must land **0 new rows** for unchanged
   sources (idempotent MERGE on `record_id` + `payload_hash`).
2. Bronze row counts reconcile with source System Tables inside the ingested window.
3. Checkpoints exist for every succeeded source and only advance after commit.
4. Freshness SLO rows are populated.

In [0]:
import sys
from pathlib import Path

REPO = Path("/Workspace/Users/dhirajsardal@gmail.com/databricks-finops-starter")
sys.path.insert(0, str(REPO / "src"))

from finops.pipeline import run_pipeline  # noqa: E402

run1 = run_pipeline(spark, conf_root=REPO / "conf", environment="dev")
display(run1)

In [0]:
run2 = run_pipeline(spark, conf_root=REPO / "conf", environment="dev")

idempotency = {
    sid: {
        "run1_inserted": r1.get("rows_inserted"),
        "run2_inserted": r2.get("rows_inserted"),
    }
    for (sid, r1), (_, r2) in zip(run1["sources"].items(), run2["sources"].items())
    if r1.get("status") == "SUCCEEDED"
}
display(idempotency)
assert all(v["run2_inserted"] == 0 for v in idempotency.values()), \
    "second run inserted rows for unchanged data — merge is not idempotent"

In [0]:
CATALOG = run1["regional_catalog"]

checks = {}

checks["bronze_usage_rows"] = spark.sql(
    f"SELECT count(*) AS n FROM {CATALOG}.bronze.billing_usage_raw"
).collect()[0].n
checks["system_usage_rows"] = spark.sql(
    "SELECT count(*) AS n FROM system.billing.usage"
).collect()[0].n
checks["checkpoints"] = spark.sql(
    f"""SELECT count(DISTINCT source_id) AS n
        FROM {CATALOG}.ops.source_checkpoint WHERE layer = 'bronze'"""
).collect()[0].n
checks["freshness_rows"] = spark.sql(
    f"SELECT count(*) AS n FROM {CATALOG}.ops.freshness_status"
).collect()[0].n
checks["quarantine_total"] = spark.sql(
    f"""SELECT coalesce(sum(failed_count), 0) AS n
        FROM {CATALOG}.ops.data_quality_result"""
).collect()[0].n

display(checks)
assert checks["bronze_usage_rows"] == checks["system_usage_rows"], \
    "bronze must reconcile exactly with system.billing.usage (initial full read)"
assert checks["checkpoints"] > 0, "bronze checkpoints missing"
assert checks["freshness_rows"] > 0, "freshness SLO rows missing"
print("ALL VALIDATION CHECKS PASSED")